# Alchemy GeomML + TDA — Финальные графики v34

In [ ]:
import os
if not os.path.exists('alchemy-geom-tda'):
    !git clone https://github.com/ThomasMoore25/alchemy-geom-tda.git
%cd alchemy-geom-tda
!git pull

In [ ]:
import glob, numpy as np, pandas as pd
import matplotlib.pyplot as plt

FIG_DIR = 'results/figures/v34'
os.makedirs(FIG_DIR, exist_ok=True)

histories = {}
for d in ['batch_size_256', 'batch_size_512', 'batch_size_1024']:
    path = f'results/experiments/{d}'
    for csv in sorted(glob.glob(f'{path}/history_*.csv')):
        name = os.path.basename(csv).replace('history_', '').rsplit('_', 2)[0]
        df = pd.read_csv(csv)
        histories[name] = df
        print(f'{d}/{name}: {len(df)} epochs')

In [ ]:
# 1. Training Curves (v34)
fig, axes = plt.subplots(2, 2, figsize=(16, 12), constrained_layout=True)
fig.suptitle('Training Curves — v34 (bs=1024, 2 GPU T4)', fontsize=16, fontweight='bold')

models = {'egnn': ('EGNN', '#2196F3'), 'egnn_tda': ('EGNN+TDA', '#FF9800'),
          'egnn_vector': ('EGNN Vector', '#4CAF50'), 'egnn_vector_tda': ('EGNN Vector+TDA', '#9C27B0'),
          'egnn_tensor_tda': ('EGNN Tensor+TDA', '#F44336')}

for ax, (col, title) in zip(axes.flat, [('val_loss', 'Val Loss'), ('val_mu_mae', 'μ MAE (Debye)'),
                                          ('val_alpha_mae', 'α MAE (Bohr³)'), ('val_gap_mae', 'gap MAE (Ha)')]):
    for key, (label, color) in models.items():
        if key in histories and col in histories[key].columns:
            ax.plot(histories[key]['epoch'], histories[key][col], color=color, label=label, linewidth=2)
    ax.set_title(title, fontsize=13); ax.set_xlabel('Epoch')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.savefig(f'{FIG_DIR}/01_training_curves.png', dpi=150, facecolor='white')
plt.show()

In [ ]:
# 2. Test Metrics Bar Chart
results = [
    ('FCNN', 0.851, 2.271, 0.0261, 1.235, 53171),
    ('SchNet', 0.131, 0.442, 0.0033, 0.186, 319491),
    ('EGNN', 0.284, 0.583, 0.0058, 0.344, 951759),
    ('EGNN+TDA', 0.298, 0.619, 0.0061, 0.362, 971831),
    ('EGNN\nVector', 4.123, 0.354, 0.0044, 0.167, 968023),
    ('EGNN\nVector+TDA', 4.121, 0.510, 0.0055, 0.216, 981439),
    ('EGNN\nTensor', 4.121, 0.957, 0.0091, 0.394, 941900),
    ('EGNN\nTensor+TDA', 4.111, 1.011, 0.0104, 0.428, 948660),
]

fig, axes = plt.subplots(1, 4, figsize=(24, 6), constrained_layout=True)
fig.suptitle('Test Metrics — All 8 Models (v34)', fontsize=16, fontweight='bold')
colors = ['#607D8B', '#795548', '#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336', '#E91E63']
labels = [r[0] for r in results]

for ax, (idx, title, unit) in zip(axes, [(1, 'μ MAE', 'D'), (2, 'α MAE', 'Bohr³'), (3, 'gap MAE', 'Ha'), (4, 'Test Loss', 'norm')]):
    vals = [r[idx] for r in results]
    bars = ax.bar(range(len(vals)), vals, color=colors, edgecolor='black')
    ax.set_xticks(range(len(vals))); ax.set_xticklabels(labels, fontsize=9, rotation=45, ha='right')
    ax.set_title(f'{title} ({unit})', fontsize=12); ax.grid(True, alpha=0.3, axis='y')
    bars[vals.index(min(vals))].set_edgecolor('gold'); bars[vals.index(min(vals))].set_linewidth(3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01, f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.savefig(f'{FIG_DIR}/02_test_metrics_bar.png', dpi=150, facecolor='white')
plt.show()

In [ ]:
# 3. Parameter Count
fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
models = ['FCNN', 'SchNet', 'EGNN', 'EGNN+TDA', 'EGNN\nVector', 'EGNN\nVector+TDA', 'EGNN\nTensor', 'EGNN\nTensor+TDA']
params = [53171, 319491, 951759, 971831, 968023, 981439, 941900, 948660]
colors = ['#607D8B', '#795548', '#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#F44336', '#E91E63']
bars = ax.barh(models, params, color=colors, edgecolor='black')
ax.set_xlabel('Parameters', fontsize=13)
ax.set_title('Model Parameter Count', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars, params):
    ax.text(bar.get_width() + 10000, bar.get_y() + bar.get_height()/2, f'{val:,}', ha='left', va='center', fontsize=10, fontweight='bold')
ax.set_xlim(0, max(params) * 1.15)
plt.savefig(f'{FIG_DIR}/03_parameter_count.png', dpi=150, facecolor='white')
plt.show()

In [ ]:
# 4. TDA Analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
fig.suptitle('TDA Impact: Why TDA Degrades Performance', fontsize=14, fontweight='bold')

ax = axes[0]
cats = ['H₀ Betti', 'H₁ Betti', 'H₂ Betti', 'Entropy+Pers']
total = [16, 16, 16, 4]; zero = [0, 12, 16, 0]; nonzero = [t-z for t,z in zip(total, zero)]
ax.bar(cats, nonzero, color='#4CAF50', label='Informative', edgecolor='black')
ax.bar(cats, zero, bottom=nonzero, color='#F44336', label='Always zero', edgecolor='black')
ax.set_title('TDA Feature Quality (52D)', fontsize=12); ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.text(0.5, 0.95, '54% ALWAYS ZERO', transform=ax.transAxes, ha='center', fontsize=14, fontweight='bold', color='#F44336', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax = axes[1]
metrics = ['μ MAE', 'α MAE', 'gap MAE', 'test_loss']
egnn = [0.284, 0.583, 0.0058, 0.344]; tda = [0.298, 0.619, 0.0061, 0.362]
x = np.arange(len(metrics)); w = 0.35
ax.bar(x - w/2, egnn, w, label='EGNN', color='#2196F3', edgecolor='black')
ax.bar(x + w/2, tda, w, label='EGNN+TDA', color='#FF9800', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_title('EGNN vs EGNN+TDA (v34)', fontsize=12); ax.legend(); ax.grid(True, alpha=0.3, axis='y')
for i, (e, t) in enumerate(zip(egnn, tda)):
    ax.annotate(f'+{(t-e)/e*100:.0f}%', xy=(i+w/2, t), xytext=(i+w/2, t*1.1), ha='center', fontsize=10, color='#F44336', fontweight='bold')

plt.savefig(f'{FIG_DIR}/04_tda_analysis.png', dpi=150, facecolor='white')
plt.show()

In [ ]:
# 5. EGNN vs EGNN Vector
fig, axes = plt.subplots(1, 4, figsize=(20, 5), constrained_layout=True)
fig.suptitle('EGNN (scalar μ) vs EGNN Vector (vector μ)', fontsize=14, fontweight='bold')

for ax, (col, title) in zip(axes, [('val_loss', 'Val Loss'), ('val_mu_mae', 'μ MAE (Debye)'), ('val_alpha_mae', 'α MAE (Bohr³)'), ('val_gap_mae', 'gap MAE (Ha)')]):
    if 'egnn' in histories and col in histories['egnn'].columns:
        ax.plot(histories['egnn']['epoch'], histories['egnn'][col], color='#2196F3', label='EGNN', linewidth=2)
    if 'egnn_vector' in histories and col in histories['egnn_vector'].columns:
        ax.plot(histories['egnn_vector']['epoch'], histories['egnn_vector'][col], color='#4CAF50', label='EGNN Vector', linewidth=2)
    ax.set_title(title, fontsize=12); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True, alpha=0.3)

plt.savefig(f'{FIG_DIR}/05_egnn_vs_vector.png', dpi=150, facecolor='white')
plt.show()

In [ ]:
# 6. Batch Size Impact
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
bs = [256, 512, 1024]; mu = [0.179, 0.245, 0.284]; alpha = [0.393, 0.460, 0.583]; gap = [0.0041, 0.0046, 0.0058]
ax.plot(bs, mu, 'o-', color='#2196F3', label='μ MAE (Debye)', linewidth=2, markersize=10)
ax.plot(bs, alpha, 's-', color='#FF9800', label='α MAE (Bohr³)', linewidth=2, markersize=10)
ax.plot(bs, gap, '^-', color='#4CAF50', label='gap MAE (Ha)', linewidth=2, markersize=10)
ax.set_xlabel('Batch Size', fontsize=13); ax.set_ylabel('MAE (lower = better)', fontsize=13)
ax.set_title('Batch Size Impact on EGNN', fontsize=14, fontweight='bold')
ax.set_xticks(bs); ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
for b, m, a, g in zip(bs, mu, alpha, gap):
    ax.annotate(f'{m:.3f}', (b, m), textcoords='offset points', xytext=(0, 12), ha='center', fontsize=10)
    ax.annotate(f'{a:.3f}', (b, a), textcoords='offset points', xytext=(0, 12), ha='center', fontsize=10)
    ax.annotate(f'{g:.4f}', (b, g), textcoords='offset points', xytext=(0, 12), ha='center', fontsize=10)
plt.savefig(f'{FIG_DIR}/06_batch_size_impact.png', dpi=150, facecolor='white')
plt.show()